# AtlasInfer on Kaggle — benchmark bigger models

Runs the full AtlasInfer suite (perplexity/memory, head-to-head vs bitsandbytes, runtime) on GPUs bigger than a 6 GB laptop, so the trends can be checked at real model scale.

## Setup (do this first)
1. **Settings -> Accelerator -> GPU T4 x2** (32 GB total) or **GPU P100** (16 GB). Use **T4** if you want the optional Triton-kernel cell (Turing+).
2. **Settings -> Internet -> On** (needed to download models).
3. *(Optional, for gated models like Llama/Gemma)* **Add-ons -> Secrets -> add `HF_TOKEN`** (a read token from https://huggingface.co/settings/tokens).

## Model-size guide (the fp16 baseline must fit in VRAM)
- **One 16 GB GPU:** ~<=4B comfortable (Qwen2.5-3B, Gemma-3-4B, Phi-4-mini); ~7B is tight.
- **T4 x2 (32 GB):** set `DEVICE_MAP = '--device-map'` below to shard a 7-13B model across both GPUs.

In [ ]:
# 1) Clone + install AtlasInfer
!git clone -q https://github.com/nishantkluhera/AtlasInfer.git
%cd AtlasInfer
!pip install -q -e '.[benchmark]' bitsandbytes

In [ ]:
# 2) Hugging Face auth (optional; only needed for gated models). Uses a Kaggle Secret named HF_TOKEN.
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF token loaded -- gated models (Llama/Gemma) available')
except Exception as e:
    print('No HF_TOKEN secret; gated models will 403. Open models (Qwen / Mistral / Pythia) still work.')
import torch
print('GPUs visible:', torch.cuda.device_count(), '|', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

## Pick a model

In [ ]:
MODEL = 'Qwen/Qwen2.5-3B'   # comfy on one 16 GB GPU. Bigger: 'Qwen/Qwen2.5-7B', 'mistralai/Mistral-7B-v0.3'
EVAL_TOKENS = 40000
DEVICE_MAP = ''            # set to '--device-map' to shard a >16 GB model across T4 x2

In [ ]:
# 3) Perplexity / memory sweep (writes results/<model>.md/.json/.png)
!python benchmark.py --model {MODEL} --eval-tokens {EVAL_TOKENS} --bits 4.5 5 6 7 {DEVICE_MAP}

In [ ]:
# 4) Head-to-head vs bitsandbytes (INT8, NF4, GPTQ-NF4, mixed)
!python compare_baselines.py --model {MODEL} --eval-tokens {EVAL_TOKENS} {DEVICE_MAP}

In [ ]:
# 5) Runtime: peak GPU memory + decode tok/s (single-GPU; no --device-map here)
!python bench_latency.py --model {MODEL}

## Optional: fused Triton kernel microbenchmark (needs a **T4** / Turing+ GPU; P100 won't compile it)

In [ ]:
!python bench_triton_kernel.py

## Download results

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/atlasinfer_results', 'zip', 'results')
print('Saved /kaggle/working/atlasinfer_results.zip -- download it from the Output tab.')